# Проверка RTMDet-s: val и все stamps

Здесь два разных результата:

1. **Val — 145 фото:** 51 `stamps` и 94 негатива. Эти кадры не используются при обучении.
   Оценка общим COCO-оценщиком, precision/recall и ложные срабатывания на негативах.
2. **Все stamps — 254 фото:** галерея рамок на всех размеченных фотографиях клейм.
   Включает 203 train и 51 val фото, поэтому её метрики не являются независимой оценкой.

Используются лучшие веса **после полного обучения на 50 эпох**. Негативы не выбираются
повторно. Порог берётся из оценки выбранного checkpoint на val; на всех stamps он
не подбирается заново. Пересечения предметов между позитивами и негативами разных
частей описаны в manifest; отдельного независимого test в данных пока нет.

Ядро: **CeramicNet RTMDet (CPU)** из тренировочного ноутбука. Это нативный запуск
MMDetection с `.pth`, экспорт в ONNX не нужен. Общий YOLO-бенчмарк не меняется.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.util
import json
import os
import sys

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "detection_pipeline").is_dir()), None)
if PROJECT is None:
    raise RuntimeError("Запусти ноутбук внутри CeramicNet из ветки pipeline")
sys.path.insert(0, str(PROJECT))
IMAGES = Path(os.environ.get("CERAMIC_IMAGES", str(PROJECT.parent / "Клеймы"))).expanduser().resolve()
SPLITS = PROJECT / "detection_pipeline/splits/v3_seed42_reserve80_active15"
MMDET_REPO = Path(os.environ.get("MMDET_REPO", str(PROJECT / ".local/rtmdet/mmdetection"))).expanduser().resolve()
# Если нужен конкретный опыт, укажи его папку вместо None.
TRAIN_RUN = None
CURRENT_MANIFEST = json.loads((SPLITS / "manifest.json").read_text())

if TRAIN_RUN is None:
    completed = []
    for candidate in sorted((PROJECT / "runs").glob("rtmdet_s_*")):
        tracking = candidate / "train/tracking/run.json"
        metadata = candidate / "configs/rtmdet_s.metadata.json"
        if tracking.is_file() and metadata.is_file() and (candidate / "val/metrics.json").is_file():
            info = json.loads(metadata.read_text())
            if (json.loads(tracking.read_text())["status"] == "completed"
                    and info["epochs"] >= 50
                    and info.get("split_fingerprint") == CURRENT_MANIFEST["fingerprint"]):
                completed.append(candidate)
    if not completed:
        raise RuntimeError("Сначала дождись полного обучения RTMDet-s на 50 эпох. Пробные веса после 1 эпохи не выбираются.")
    TRAIN_RUN = completed[-1]
TRAIN_RUN = Path(TRAIN_RUN).resolve()
run_info = json.loads((TRAIN_RUN / "configs/rtmdet_s.metadata.json").read_text())
if run_info.get("split_fingerprint") != CURRENT_MANIFEST["fingerprint"]:
    raise RuntimeError("Эти веса обучены на другом разбиении; выбери завершённый опыт v3")
CONFIG = TRAIN_RUN / "configs/rtmdet_s.py"
checkpoints = list((TRAIN_RUN / "train").glob("best_coco_bbox_mAP_*.pth"))
if len(checkpoints) != 1:
    raise RuntimeError(f"Не найден единственный лучший checkpoint: {checkpoints}")
CHECKPOINT = checkpoints[0]
VALIDATION_METRICS = TRAIN_RUN / "val/metrics.json"
stamp = datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT = TRAIN_RUN / f"test_{stamp}"
print("Опыт:", TRAIN_RUN)
print("Лучшие веса:", CHECKPOINT)
print("Результаты:", OUTPUT)

## 1. Проверяем наборы перед запуском

Отбор повторяется только как чтение сохранённых записей JSON: val содержит
ровно 145 фото и 94 негатива, все stamps — ровно 254 фото. Нового random split нет.

In [ ]:
processor_file = PROJECT / "test_pipelines/detection_models/RTMDet-s/test_rtmdet_s.py"
spec = importlib.util.spec_from_file_location("rtmdet_s_benchmark", processor_file)
benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark)
for scope in ("val", "stamps"):
    data, description = benchmark.evaluation_data(SPLITS, scope)
    print(scope, description)

## 2. Оценка на val

Сохраняются `predictions.json`, `metrics.json`, время обработки, CSV и галерея.
Измерение времени включает resize/format, модель и NMS; загрузка модели,
первый прогревочный вызов и чтение фото с диска исключаются.

In [ ]:
val_metrics = benchmark.run(
    config=CONFIG, checkpoint=CHECKPOINT, images_root=IMAGES,
    splits=SPLITS, repo=MMDET_REPO, output=OUTPUT / "val",
    scope="val", validation_metrics=VALIDATION_METRICS, device="cpu")

## 3. Все фото stamps

Сохраняем рамки для просмотра на **каждом** фото. Зелёный — разметка, красный —
предсказание выше фиксированного на val порога. Галерея включает train;
для отчёта о качестве используйте результат из предыдущей ячейки.

In [ ]:
stamps_metrics = benchmark.run(
    config=CONFIG, checkpoint=CHECKPOINT, images_root=IMAGES,
    splits=SPLITS, repo=MMDET_REPO, output=OUTPUT / "stamps",
    scope="stamps", validation_metrics=VALIDATION_METRICS, device="cpu")

In [ ]:
from IPython.display import FileLink, display
for scope in ("val", "stamps"):
    print(scope)
    display(FileLink(str(OUTPUT / scope / "gallery.html")))
    display(FileLink(str(OUTPUT / scope / "benchmark_results.csv")))
    display(FileLink(str(OUTPUT / scope / "metrics.json")))

## Сравнение с другими моделями

Результаты лежат отдельно для val и всех stamps; каждый CSV содержит размер набора,
число тренировочных кадров, хеш разметки и checkpoint, порог и устройство.
Сравнивать качество моделей нужно на одном и том же сохранённом val и общим оценщиком.
Старый `benchmark_results.csv` у YOLO использует другое разбиение и ONNX Runtime,
поэтому добавлять туда эти числа как прямое сравнение некорректно.

В Git хранятся код и ноутбуки. Фотографии, веса и локальные галереи остаются в `runs/`.
Инструкция и команды запуска — в README этой папки.